In [24]:
import os
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from data.dataloader_phased import StreamingParquetBuffer, WindowSpec

In [25]:
root = "/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data"
data_dir = os.path.join(root, "data")

parquet_files = []
for r, _, fs in os.walk(data_dir):
    for f in fs:
        if f.endswith(".parquet"):
            parquet_files.append(os.path.join(r, f))
parquet_files = sorted(parquet_files)
print(f"Found {len(parquet_files)} .parquet files under {data_dir}")

if len(parquet_files) == 0:
    raise FileNotFoundError(f"No .parquet files found under {data_dir}")

Found 6 .parquet files under /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data


In [26]:
# print the first few lines of the first parquet file to verify it's readable
import pandas as pd
df = pd.read_parquet(parquet_files[0])
print("First few lines of the first parquet file:")
# print(df.head())
print(df[-10:]) 
print(len(df))

First few lines of the first parquet file:
                           voltage_l1_116  voltage_l2_116  voltage_l3_116  \
timestamp_dst                                                               
2024-05-28 09:30:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 09:45:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 10:00:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 10:15:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 10:30:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 10:45:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 11:00:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 11:15:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 11:30:00+00:00            <NA>            <NA>            <NA>   
2024-05-28 11:45:00+00:00            <NA>            <NA>            <NA>   

                           activ

In [32]:
# Visualize ALL parquet files (one preview plot per file)
# using 15-minute data, skipping the first N days.

out_dir = os.path.join(root, "output", "parquet_previews")
os.makedirs(out_dir, exist_ok=True)

skip_days = 14
window_len_steps = 96   # 24 hours at 15-minute resolution
stride_steps = 96       # move by one full day
find_first_nonzero_after_skip = True
max_probe_days = 120

saved = 0
for idx, parquet_path in enumerate(parquet_files):
    try:
        ds_one = StreamingParquetBuffer(
            sm_files=[parquet_path],
            replay_buffer_capacity=256,
            min_buffer_size=8,
            window_spec=WindowSpec(input_len=window_len_steps, stride=stride_steps),
            normalizer=None,
            num_workers=0,
        )
        loader_one = DataLoader(ds_one, batch_size=1, num_workers=0)
        loader_iter = iter(loader_one)

        # Skip first N daily windows
        for _ in range(skip_days):
            _ = next(loader_iter)

        # Pick first window after skip; optionally move forward until data is non-zero
        batch = next(loader_iter)
        probe = 0
        while True:
            x_one, _ = batch
            sample = x_one[0].detach().cpu().numpy()

            if sample.ndim != 2:
                raise ValueError(f"Unexpected sample shape: {sample.shape}")

            # Convert to [L, F]
            sample_lf = sample.T if sample.shape[0] <= sample.shape[1] else sample

            if (not find_first_nonzero_after_skip) or np.any(np.abs(sample_lf) > 1e-8):
                break

            probe += 1
            if probe > max_probe_days:
                raise StopIteration
            batch = next(loader_iter)

        L, F = sample_lf.shape
        labels = ["L1", "L2", "L3"] if F == 3 else [f"Feature {i+1}" for i in range(F)]

        fig, axes = plt.subplots(F, 1, figsize=(10, 2.8 * F), sharex=True)
        if F == 1:
            axes = [axes]

        # Generate clock-of-day times for x-axis (15-min intervals)
        import datetime
        import matplotlib.dates as mdates
        times = [datetime.datetime(2000,1,1,h,m) for h in range(24) for m in range(0, 60, 15)]
        if len(times) != window_len_steps:
            # fallback: linspace between 00:00 and 24:00
            times = [datetime.datetime(2000,1,1) + datetime.timedelta(minutes=15*i) for i in range(window_len_steps)]

        shown_day = skip_days + probe + 1
        for i in range(F):
            axes[i].plot(times, sample_lf[:, i], marker="o", linewidth=1, markersize=3)
            axes[i].set_title(f"{os.path.basename(parquet_path)} | day {shown_day} | {labels[i]}")
            axes[i].set_ylabel("kWh")
            axes[i].grid(alpha=0.3)

        # Format x-axis as clock time
        axes[-1].set_xlabel("Time of day")
        for ax in axes:
            ax.set_xlim(times[0], times[-1])
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
            ax.xaxis.set_minor_locator(mdates.MinuteLocator(byminute=[0,15,30,45]))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

        plt.tight_layout()

        save_path = os.path.join(out_dir, f"preview_{idx:04d}.png")
        plt.savefig(save_path, dpi=130)
        plt.close(fig)
        saved += 1

    except StopIteration:
        print(f"Skipped {os.path.basename(parquet_path)}: no non-zero daily window found after day {skip_days}")
    except Exception as e:
        print(f"Skipped {os.path.basename(parquet_path)}: {e}")

print(f"Saved {saved} preview plots to: {out_dir}")
print(f"Settings: 15-min data, daily windows ({window_len_steps} steps), skipped first {skip_days} days.")
print(f"find_first_nonzero_after_skip = {find_first_nonzero_after_skip}")

Saved 6 preview plots to: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/output/parquet_previews
Settings: 15-min data, daily windows (96 steps), skipped first 14 days.
find_first_nonzero_after_skip = True


sm_116: 
- phase 1: --
- phase 2: yes 
- phase 3: -- 

sm_17: 
- phase 1: yes
- phase 2: yes 
- phase 3: -- 

sm_40: 
- phase 1: yes
- phase 2: yes 
- phase 3: yes

sm_87: 
- phase 1: yes
- phase 2: yes 
- phase 3: --

## Where do reconstruction samples come from?


The visualizer calls `next(iter(test_loader))` and picks the **first `num_samples`** from that single batch.  
The cell below loads a batch from all parquet files together (the same way training does) and shows for each sample:
- which meter it comes from
- the day/time
- which phases carry actual data (non-zero sum)

In [9]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# --- Build a combined loader from ALL parquet files (same as training does) ---
ds_all = StreamingParquetBuffer(
    sm_files=parquet_files,          # all files
    replay_buffer_capacity=512,
    min_buffer_size=8,
    window_spec=WindowSpec(input_len=24, stride=24),
    normalizer=None,
    num_workers=0,
)
loader_all = DataLoader(ds_all, batch_size=16, num_workers=0)

# Grab exactly one batch  (this is what the reconstruction visualizer sees)
x_batch, cond_batch = next(iter(loader_all))

# cond shape: [B, 2, L]  -> cond[:,0,0] = meter_id,  cond[:,1,0] = timestamp
meter_ids  = cond_batch[:, 0, 0].long().tolist()
timestamps = cond_batch[:, 1, 0].long().tolist()

# x shape: [B, F, L]  (phase-first) or [B, L, F]
# work out which axis is length 24
x_np = x_batch.detach().cpu().numpy()
if x_np.shape[1] == 24:          # [B, L, F]
    x_lf = x_np                   # already [B, L, F]
elif x_np.shape[2] == 24:        # [B, F, L]
    x_lf = x_np.transpose(0, 2, 1)  # -> [B, L, F]
else:
    x_lf = x_np

n_features = x_lf.shape[2]
phase_names = ["L1", "L2", "L3"] if n_features == 3 else [f"F{i+1}" for i in range(n_features)]

# --- Print a summary table ---
print(f"{'#':<4} {'meter_id':<12} {'date (UTC)':<22} ", end="")
for p in phase_names:
    print(f"{'has_'+p:<10}", end="")
print()
print("-" * (4 + 12 + 22 + 10 * n_features))

rows = []
for i, (m_id, ts) in enumerate(zip(meter_ids, timestamps)):
    dt = pd.Timestamp(ts, unit='s')
    phase_active = ["YES" if x_lf[i, :, f].sum() > 0.01 else "---" for f in range(n_features)]
    row = [i + 1, m_id, str(dt)[:19]] + phase_active
    rows.append(row)
    print(f"{i+1:<4} {m_id:<12} {str(dt)[:19]:<22} ", end="")
    for a in phase_active:
        print(f"{a:<10}", end="")
    print()

print(f"\nBatch size: {len(meter_ids)} samples")
print(f"Reconstruction plot shows samples 1 – num_samples (default 5)")

#    meter_id     date (UTC)             has_L1    has_L2    has_L3    
--------------------------------------------------------------------
1    116          2023-09-04 08:45:00    ---       YES       ---       
2    116          2023-09-05 20:45:00    ---       YES       ---       
3    116          2023-09-05 20:45:00    ---       YES       ---       
4    116          2023-09-05 14:45:00    ---       YES       ---       
5    116          2023-09-04 20:45:00    ---       YES       ---       
6    116          2023-09-05 20:45:00    ---       YES       ---       
7    116          2023-09-07 02:45:00    ---       YES       YES       
8    116          2023-09-06 20:45:00    ---       YES       ---       
9    116          2023-09-07 14:45:00    ---       YES       ---       
10   116          2023-09-08 02:45:00    ---       YES       YES       
11   116          2023-09-08 08:45:00    ---       YES       ---       
12   116          2023-09-05 02:45:00    ---       YES       YES   